In [1]:
# ----------------------------------
# 
# postprocess time dynamics
# 
# ----------------------------------
import fnmatch
import glob
import os
import pickle
import sys
import re
import tempfile
from typing import Tuple

import cmocean.cm as cmo
import fsspec         # for AWS integration
import s3fs
from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D  # for custom legend entries (needed for contour plot)
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# --- read in cdr calculation functions  
sys.path.append(os.path.abspath('/home/tykukla/ew-workflows/run_scepter'))
import cdr_fxns_postproc as cfp
# ---


# --- Decide which CDR calculations to perform
cdr_calc_list = ["co2_flx",       
                 "camg_flx",
                 "totcat_flx",
                 # "carbalk_flx", # (turn this back on when the runs are re-run with fixed cflx file)
                 "rockdiss"
                ]
# ---
savehere = "s3://carbonplan-carbon-removal/SCEPTER/scepter_output_proc/"

In [2]:
# ************************************************
# batchfile name 
timestep = "hourly"
batchname = f"meanAnn_gbas_shortRun_timeDynamics_test_{timestep}_noFert_gs+apprate_v0.csv"
dustsp = "gbas"
multiyear = False
runtype = "field"
# set the time_horizon for computing the integrated flux 
time_horizon = 1.
# ************************************************

# --- location of data (outdir) and batchfile (csv_loc)
outdir = "s3://carbonplan-carbon-removal/SCEPTER/scepter_output_scratch/"
csv_loc = "s3://carbonplan-carbon-removal/ew-workflows-data/scepter/batch"

In [3]:
# --- read and tidy up the batch file 
dfin = pd.read_csv(os.path.join(csv_loc, batchname))

# add column for the full run id
if multiyear:
    dfin["newrun_id_full"] = dfin['newrun_id'] + f"_composite_{runtype}"
else:
    dfin["newrun_id_full"] = dfin['newrun_id'] + "_" + dfin['dustsp'] + "_" + runtype + "_tau"+dfin["duration"].astype(float).astype(str).str.replace(".", "p")  # (duration has to be turned into float first because otherwise we miss the decimal pt)

# identify the control runs
# ( assume ctrl means dustrate == 0 )
dfin['ctrl_run'] = np.where(dfin['dustrate'] == 0, True, False)


# add a column for the dustrate in ton_ha_yr
if "dustrate" in dfin.columns:
    dfin["dustrate_ton_ha_yr"] = dfin["dustrate"] / 100 
dfin

,dustrate,dustrad,site,spinrun,climatefiles,dust_ts_fn,duration,dustsp,dustsp_2nd,dustrate_2nd,...,use_psdrain_datfile,include_roughness_sa,aws_save,aws_bucket,scepter_exec_name,newrun_id,psdrain_meanRad,newrun_id_full,ctrl_run,dustrate_ton_ha_yr
0,0.0,5,albany_hourly,site_311a_pr9_spintuneup4,albany_hourly,gbas_15yr_1app_no2nd_001.csv,1,gbas,amnt,0.0,...,False,True,move,s3://carbonplan-carbon-removal/SCEPTER/scepter...,scepter,noFert_gbas_timeDynamics_test_hourly_albany_ho...,0.000005,noFert_gbas_timeDynamics_test_hourly_albany_ho...,True,0.0
1,0.0,5,atlanta_hourly,site_311a_pr9_spintuneup4,atlanta_hourly,gbas_15yr_1app_no2nd_001.csv,1,gbas,amnt,0.0,...,False,True,move,s3://carbonplan-carbon-removal/SCEPTER/scepter...,scepter,noFert_gbas_timeDynamics_test_hourly_atlanta_h...,0.000005,noFert_gbas_timeDynamics_test_hourly_atlanta_h...,True,0.0
2,0.0,5,central_valley_hourly,site_311a_pr9_spintuneup4,central_valley_hourly,gbas_15yr_1app_no2nd_001.csv,1,gbas,amnt,0.0,...,False,True,move,s3://carbonplan-carbon-removal/SCEPTER/scepter...,scepter,noFert_gbas_timeDynamics_test_hourly_central_v...,0.000005,noFert_gbas_timeDynamics_test_hourly_central_v...,True,0.0
3,0.0,5,minneapolis_hourly,site_311a_pr9_spintuneup4,minneapolis_hourly,gbas_15yr_1app_no2nd_001.csv,1,gbas,amnt,0.0,...,False,True,move,s3://carbonplan-carbon-removal/SCEPTER/scepter...,scepter,noFert_gbas_timeDynamics_test_hourly_minneapol...,0.000005,noFert_gbas_timeDynamics_test_hourly_minneapol...,True,0.0
4,100.0,5,albany_hourly,site_311a_pr9_spintuneup4,albany_hourly,gbas_15yr_1app_no2nd_001.csv,1,gbas,amnt,0.0,...,False,True,move,s3://carbonplan-carbon-removal/SCEPTER/scepter...,scepter,noFert_gbas_timeDynamics_test_hourly_albany_ho...,0.000005,noFert_gbas_timeDynamics_test_hourly_albany_ho...,False,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,2000.0,200,minneapolis_hourly,site_311a_pr9_spintuneup4,minneapolis_hourly,gbas_15yr_1app_no2nd_001.csv,1,gbas,amnt,0.0,...,False,True,move,s3://carbonplan-carbon-removal/SCEPTER/scepter...,scepter,noFert_gbas_timeDynamics_test_hourly_minneapol...,0.000200,noFert_gbas_timeDynamics_test_hourly_minneapol...,False,20.0
64,4000.0,5,minneapolis_hourly,site_311a_pr9_spintuneup4,minneapolis_hourly,gbas_15yr_1app_no2nd_001.csv,1,gbas,amnt,0.0,...,False,True,move,s3://carbonplan-carbon-removal/SCEPTER/scepter...,scepter,noFert_gbas_timeDynamics_test_hourly_minneapol...,0.000005,noFert_gbas_timeDynamics_test_hourly_minneapol...,False,40.0
65,4000.0,50,minneapolis_hourly,site_311a_pr9_spintuneup4,minneapolis_hourly,gbas_15yr_1app_no2nd_001.csv,1,gbas,amnt,0.0,...,False,True,move,s3://carbonplan-carbon-removal/SCEPTER/scepter...,scepter,noFert_gbas_timeDynamics_test_hourly_minneapol...,0.000050,noFert_gbas_timeDynamics_test_hourly_minneapol...,False,40.0
66,4000.0,100,minneapolis_hourly,site_311a_pr9_spintuneup4,minneapolis_hourly,gbas_15yr_1app_no2nd_001.csv,1,gbas,amnt,0.0,...,False,True,move,s3://carbonplan-carbon-removal/SCEPTER/scepter...,scepter,noFert_gbas_timeDynamics_test_hourly_minneapol...,0.000100,noFert_gbas_timeDynamics_test_hourly_minneapol...,False,40.0


In [4]:
# --- decide which columns from dfin we want to keep when we 
#     create our flux dicts below
# 
# these should be the columns that become dimensions in the 
# later xr datasets... 
# 
dfin_cols_to_keep = ["dustrad", "dustrate_ton_ha_yr", "site"]
dfin_cols_to_keep_time = ["dustrad", "dustrate_ton_ha_yr", "time", "site"]

In [5]:
# --- read in the postprocessed flux data for sil 
# (both transient and time-integrated fluxes)
flx_dict_int = cfp.read_postproc_flux(
    dfin, outdir, cdr_calc_list, dfin_cols_to_keep, 
    flx_type='int_flx', rockdiss_feedstock=dustsp
)
flx_dict = cfp.read_postproc_flux(
    dfin, outdir, cdr_calc_list, dfin_cols_to_keep, 
    flx_type='flx', rockdiss_feedstock=dustsp
)

solving co2_flx
could not find noFert_gbas_timeDynamics_test_hourly_atlanta_hourly_app_0p0_psize_5_gbas_field_tau1p0 -- run 1
could not find noFert_gbas_timeDynamics_test_hourly_central_valley_hourly_app_4000p0_psize_5_gbas_field_tau1p0 -- run 48
could not find noFert_gbas_timeDynamics_test_hourly_central_valley_hourly_app_4000p0_psize_100_gbas_field_tau1p0 -- run 50


KeyboardInterrupt: 

In [6]:
# --- compute CDR fluxes for cc and sil (with losses considered)
# ---
# [ time-integrated ]
cdr_dict_int_full, cdr_dict_int_sum = cfp.cdr_int_per_group(
    flx_dict_int, time_horizon, 
    cdr_calc_list, dfin_cols_to_keep,
    bysite = True
)
# [ transient ]
cdr_dict_full, cdr_dict_sum = cfp.cdr_int_per_group(
    flx_dict, time_horizon, 
    cdr_calc_list, dfin_cols_to_keep,
    bysite = True
)

solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss


In [7]:
# --- get emissions 
dustrate_name = "dustrate_ton_ha_yr" # note, we multiply by duration in emissions_calc, so we only want annual rate here
# set the inputs ***************
p80_input = 1.3e3
truck_km = 0.1e3
barge_km = 0.0e3
barge_diesel_km = 0
Efactor_org = "MRO"
# ******************************

cdr_dict_sum['rockdiss'] = cfp.emissions_calculator_df(cdr_dict_int_sum['rockdiss'], dustrate_name, 
                                                        p80_input, truck_km, barge_km, barge_diesel_km, 
                                                        Efactor_org, mineral=dustsp)


Emissions Calculator: Cannot convert column 'site' to float: Unable to parse string "albany_daily" at position 0. Ignore if expected.


In [8]:
# --- compute CDR w/ loss rates and convert to xr datasets
outds_int = cfp.cdr_ds(
    cdr_dict = cdr_dict_int_sum, dims = dfin_cols_to_keep, cdr_calc_list = cdr_calc_list
)
# --- repeat for time-dependent time-integrated data
outds_int_time = cfp.cdr_ds(
    cdr_dict = cdr_dict_int_full, dims = dfin_cols_to_keep_time, 
    cdr_calc_list = cdr_calc_list, skip_loss = True,
)
# --- repeat for time-transient data (note dims now includes "time")
outds = cfp.cdr_ds(
    cdr_dict = cdr_dict_full, dims = dfin_cols_to_keep_time, 
    cdr_calc_list = cdr_calc_list, skip_loss = True,
)


solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss
solving co2_flx
solving camg_flx
solving totcat_flx
solving rockdiss


In [9]:
# --- save result
# set file system
s3 = s3fs.S3FileSystem(anon=False)

# filenames 
fn1 = "results_proc_transient.nc"
fn2 = "results_proc_integrated.nc"
fn3 = "results_proc_integrated+time.nc"

# ------------------
# write datasets
# ------------------
# [ time-transient ]
save1 = os.path.join(savehere, batchname, fn1)
# write locally first
with tempfile.NamedTemporaryFile(suffix=".nc") as tmp:
    outds.to_netcdf(tmp.name)
    s3.put(tmp.name, save1)

# [ time-integrated ]
save2 = os.path.join(savehere, batchname, fn2)
# write locally first
with tempfile.NamedTemporaryFile(suffix=".nc") as tmp2:
    outds_int.to_netcdf(tmp2.name)
    s3.put(tmp2.name, save2)

# [ time-integrated + time ]
save3 = os.path.join(savehere, batchname, fn3)
# write locally first
with tempfile.NamedTemporaryFile(suffix=".nc") as tmp3:
    outds_int_time.to_netcdf(tmp3.name)
    s3.put(tmp3.name, save3)


In [ ]:
# ------------------------------------------------------
# ------------------------------------------------------
# ------------------------------------------------------

# --- SCRATCH ---